# Mid Session Assignment — Week 5 (Advanced RAG)

From `RagAdvanced.ipynb`:

> **Do the following:**
> - Query transformation and expansion
> - `Use BM25 based search`
> - Apply Reranking
>
> **Compare Results**

## What this notebook does

The base notebook only ever searched the Qdrant **dense** index. Here we add a
**lexical** (BM25) retriever next to it, feed both with **transformed / expanded**
queries, fuse the result lists with **Reciprocal Rank Fusion**, and finally
**rerank** the fused pool with a stronger model.

Then we measure all of it, because "compare results" without numbers is just vibes.

| # | Stage | Implementation |
|---|-------|----------------|
| 1 | Query transformation & expansion | LLM produces a rewritten query, sub-queries, synonym terms and a metadata filter — in **one** call (free-tier quota is tiny). Deterministic rule-based fallback if the quota is gone. |
| 2 | Lexical retrieval | `rank_bm25.BM25Okapi` over recipe name + ingredients + metadata |
| 3 | Dense retrieval | Qdrant + `all-MiniLM-L6-v2` (same store as the base notebook) |
| 4 | Fusion | Reciprocal Rank Fusion (RRF) across every query variant × both retrievers |
| 5 | Reranking | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) with a LaBSE bi-encoder fallback |
| 6 | Comparison | Precision@5 / @10 against a **rule-based relevance judge**, plus overlap and latency |

### 8 pipelines are compared

`dense` · `bm25` · `dense+expansion` · `bm25+expansion` · `hybrid(RRF)` ·
`hybrid+expansion` · `hybrid+expansion+rerank` · `hybrid+expansion+filter+rerank`


---
## Setup

Run from `Week5/` with the module env: `uv run jupyter lab`.
Needs `GOOGLE_API_KEY` in `Week5/.env` (only for query transformation — everything
else is local).

In [1]:

# %pip install rank-bm25 sentence-transformers  # already in Week5/pyproject.toml

import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 60)
tqdm.pandas(desc='Generating Documents')

load_dotenv()

True

In [2]:

# --- Gemini wrapper (same strategy as RagAdvanced.ipynb) --------------------
# Free-tier quotas are per-model and small, so walk a list of models: retry the
# ones that are merely rate limited, drop a model only when its *daily* allowance
# is gone. `LLM_AVAILABLE` lets every later cell degrade gracefully.
from google import genai
from google.genai import errors as genai_errors

model_names = [
    'gemini-2.5-flash-lite',
    'gemini-3.1-flash-lite',
    'gemini-flash-lite-latest',
    'gemini-2.5-flash',
]


class GeminiModel:
    """`model.generate_content(prompt).text` on top of the current google-genai SDK."""

    def __init__(self, client, model_names, max_retries=4):
        self.client = client
        self.model_names = list(model_names)
        self.max_retries = max_retries
        self.exhausted = []

    @property
    def model_name(self):
        return self.model_names[0] if self.model_names else None

    @staticmethod
    def _quota_info(exc):
        """(only_per_day_quota_left, retry_delay_seconds) read out of the error body."""
        quota_ids, retry_delay = [], None
        details = (getattr(exc, 'details', None) or {}).get('error', {}).get('details', [])
        for detail in details:
            kind = detail.get('@type', '')
            if kind.endswith('QuotaFailure'):
                quota_ids += [v.get('quotaId', '') for v in detail.get('violations', [])]
            elif kind.endswith('RetryInfo'):
                retry_delay = float(str(detail.get('retryDelay', '0s')).rstrip('s') or 0)
        per_day_only = bool(quota_ids) and all('PerDay' in q for q in quota_ids)
        return per_day_only, retry_delay

    def generate_content(self, prompt):
        while self.model_names:
            model_name = self.model_names[0]
            for attempt in range(self.max_retries):
                try:
                    return self.client.models.generate_content(
                        model=model_name, contents=prompt,
                    )
                except (genai_errors.ServerError, genai_errors.ClientError) as exc:
                    per_day_only, retry_delay = self._quota_info(exc)
                    last_attempt = attempt == self.max_retries - 1
                    retryable = getattr(exc, 'code', None) in (429, 500, 502, 503, 504)

                    if per_day_only or (retryable and last_attempt):
                        reason = ('daily quota used up' if per_day_only
                                  else f'still failing with {exc.code}')
                        print(f'  {model_name}: {reason}, falling back to the next model')
                        self.exhausted.append(self.model_names.pop(0))
                        break
                    if not retryable:
                        raise
                    wait = retry_delay or 2 ** attempt
                    print(f'  {exc.code} from Gemini, retrying in {wait}s')
                    time.sleep(wait)

        raise RuntimeError(
            'Every Gemini model ran out of free-tier quota: '
            f'{", ".join(self.exhausted)}. Wait for the daily reset (midnight '
            'Pacific), enable billing, or use a different GOOGLE_API_KEY.'
        )


api_key = os.getenv('GOOGLE_API_KEY')
LLM_AVAILABLE = bool(api_key)
model = GeminiModel(genai.Client(api_key=api_key), model_names) if LLM_AVAILABLE else None

print('LLM available:', LLM_AVAILABLE, '| model:', model.model_name if model else '-')


def parse_json_response(text):
    """Extract JSON from an LLM reply, with or without a ```json fence."""
    cleaned = text.strip()
    if '```' in cleaned:
        block = cleaned.split('```')[1]
        if block.startswith('json'):
            block = block[len('json'):]
        cleaned = block.strip()
    return json.loads(cleaned)

LLM available: True | model: gemini-2.5-flash-lite


### Knowledge base

Same corpus and same `Document` shape as the base notebook, with one addition:
every document carries a **`doc_id`** in its metadata. Fusion needs a stable key to
line up hits coming from two different retrievers.

In [3]:

from langchain_core.documents import Document

columns = ['TranslatedRecipeName', 'TranslatedIngredients',
           'PrepTimeInMins', 'CookTimeInMins', 'TotalTimeInMins', 'Servings',
           'Cuisine', 'Course', 'Diet', 'TranslatedInstructions', 'URL',
           'ComplexityLevel', 'MainIngredient']

df = pd.read_csv('./IndianFoodDataset.csv').set_index('Srno')[columns]


def convert_to_doc(row):
    return Document(
        page_content=f"""
# Recipe Name: {row['TranslatedRecipeName']}
> URL: {row['URL']}

## Ingredients:

{row['TranslatedIngredients']}

## Instructions:

{row['TranslatedInstructions']}
""",
        metadata={
            'doc_id': int(row.name),            # <- stable key for RRF
            'TranslatedRecipeName': row['TranslatedRecipeName'],
            'PrepTimeInMins': row['PrepTimeInMins'],
            'CookTimeInMins': row['CookTimeInMins'],
            'TotalTimeInMins': row['TotalTimeInMins'],
            'Servings': row['Servings'],
            'Cuisine': row['Cuisine'],
            'Course': row['Course'],
            'Diet': row['Diet'],
            'ComplexityLevel': row['ComplexityLevel'],
            'MainIngredient': row['MainIngredient'],
        },
    )


data = df.progress_apply(convert_to_doc, axis=1).tolist()
doc_by_id = {doc.metadata['doc_id']: doc for doc in data}

print(f'{len(data)} recipe documents')
df.head(3)

Generating Documents:   0%|          | 0/6871 [00:00<?, ?it/s]

6871 recipe documents


,TranslatedRecipeName,TranslatedIngredients,PrepTimeInMins,CookTimeInMins,TotalTimeInMins,Servings,Cuisine,Course,Diet,TranslatedInstructions,URL,ComplexityLevel,MainIngredient
Srno,,,,,,,,,,,,,
1,Masala Karela Recipe,"6 Karela (Bitter Gourd/ Pavakkai) - deseeded,Salt - to t...",15,30,45,6,Indian,Side Dish,Diabetic Friendly,"To begin making the Masala Karela Recipe,de-seed the kar...",https://www.archanaskitchen.com/masala-karela-recipe,Hard,deseeded
2,Spicy Tomato Rice (Recipe),"2-1 / 2 cups rice - cooked, 3 tomatoes, 3 teaspoons BC B...",5,10,15,3,South Indian Recipes,Main Course,Vegetarian,"To make tomato puliogere, first cut the tomatoes. Now pu...",http://www.archanaskitchen.com/spicy-tomato-rice-recipe-...,Hard,cooked
3,Ragi Semiya Upma Recipe - Ragi Millet Vermicelli Breakfast,"1-1/2 cups Rice Vermicelli Noodles (Thin),1 Onion - slic...",20,30,50,4,South Indian Recipes,South Indian Breakfast,High Protein Vegetarian,"To begin making the Ragi Vermicelli Recipe, first steam ...",http://www.archanaskitchen.com/ragi-vermicelli-semiya-re...,Hard,(Thin)


In [4]:

# Embedding models + dense (Qdrant) index -- first run downloads the models
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import models as qdrant_models

model_384 = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
model_768 = HuggingFaceEmbeddings(model_name='sentence-transformers/LaBSE')

vector_store = QdrantVectorStore.from_documents(
    data,
    model_384,
    collection_name='indian-food-assignment',
    location=':memory:',
)


def to_qdrant_filter(metadata_filter):
    """{'Cuisine': 'Indian'} -> qdrant Filter. Lists become MatchAny."""
    if not metadata_filter:
        return None

    def to_condition(key, value):
        if isinstance(value, (list, tuple, set)):
            match = qdrant_models.MatchAny(any=list(value))
        else:
            match = qdrant_models.MatchValue(value=value)
        return qdrant_models.FieldCondition(key=f'metadata.{key}', match=match)

    return qdrant_models.Filter(
        must=[to_condition(k, v) for k, v in metadata_filter.items()]
    )


print('dense index ready')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dense index ready


---
## Part 1 — Query transformation and expansion

A raw user request (`"something non-veg, spicy and quick"`) is written in *user*
language; the corpus is written in *recipe* language (`"Chicken Chettinad Masala …
red chillies, curry leaves"`). Transformation closes that gap. The obvious hypothesis is that it should matter
far more for BM25 than for dense search — a lexical index can only match words
that are literally there. Part 5 measures whether that actually holds here.
(It does not; see the Findings.)

Four things are produced from one prompt (one API call, because free-tier quota is
about 20 requests/day):

1. **`rewritten`** — a compact, search-friendly restatement
2. **`subqueries`** — the request decomposed into independent facets
3. **`expansion_terms`** — culinary synonyms / ingredient names the corpus is likely to use
4. **`metadata`** — an exact-match filter dict for the Qdrant pre-filter

`QueryPlan.variants` is what the retrievers actually consume: the original query, the
rewrite, each sub-query, and one keyword bag built from the expansion terms.

In [5]:

from dataclasses import dataclass, field


@dataclass
class QueryPlan:
    original: str
    rewritten: str = ''
    subqueries: list = field(default_factory=list)
    expansion_terms: list = field(default_factory=list)
    metadata: dict = field(default_factory=dict)
    source: str = 'llm'        # 'llm' or 'rule-based'

    @property
    def variants(self):
        """Every string we will actually send to a retriever (deduped, order kept)."""
        out = [self.original]
        if self.rewritten:
            out.append(self.rewritten)
        out += list(self.subqueries)
        if self.expansion_terms:
            out.append(' '.join(self.expansion_terms))   # keyword bag, gold for BM25
        seen, deduped = set(), []
        for q in out:
            key = q.strip().lower()
            if q.strip() and key not in seen:
                seen.add(key)
                deduped.append(q.strip())
        return deduped

    def show(self):
        print(f'original        : {self.original}')
        print(f'rewritten       : {self.rewritten}')
        print(f'subqueries      : {self.subqueries}')
        print(f'expansion_terms : {self.expansion_terms}')
        print(f'metadata filter : {self.metadata}')
        print(f'source          : {self.source}  ->  {len(self.variants)} query variants')


# ---- the metadata vocabulary the LLM is allowed to filter on ----------------
FILTERABLE = ['Cuisine', 'Diet', 'Course', 'ComplexityLevel']
metadata_vocab = '\n'.join(
    f"- '{col}': {sorted(df[col].dropna().unique().tolist())[:40]}"
    for col in FILTERABLE
)

TRANSFORM_PROMPT = """
You are the query understanding step of a recipe search engine over an Indian food
dataset. Transform and expand the user request below.

user query: {query}

Return ONE valid JSON object, nothing else, with exactly these keys:

"rewritten":       string  - one compact search-friendly phrase (no filler words)
"subqueries":      array of 2-4 strings - the request split into independent facets
"expansion_terms": array of 6-12 strings - synonyms, ingredient names and cooking
                   terms an Indian recipe text would actually use for this request
                   (single words or short phrases, no duplicates of the query itself)
"metadata":        object - exact-match filters chosen ONLY from the vocabulary
                   below. Use a list as the value to allow several. Leave it {{}}
                   when the request implies nothing. Prefer FEW filters - a wrong
                   filter silently removes correct recipes.

available metadata (exact match only):
{vocab}
"""

In [6]:

# Deterministic fallback: no API key / quota gone must not break the comparison.
SYNONYMS = {
    'spicy': ['chilli', 'masala', 'green chilli', 'red chilli powder', 'pepper'],
    'quick': ['quick', 'instant', 'easy', '10 minutes', 'no cook'],
    'healthy': ['healthy', 'nutritious', 'low calorie', 'steamed'],
    'non oily': ['steamed', 'roasted', 'grilled', 'less oil'],
    'non vegetarian': ['chicken', 'mutton', 'fish', 'prawn', 'egg'],
    'vegetarian': ['paneer', 'dal', 'vegetable', 'sabzi'],
    'breakfast': ['idli', 'dosa', 'upma', 'poha', 'paratha'],
    'dessert': ['halwa', 'kheer', 'ladoo', 'barfi', 'payasam'],
    'sweet': ['jaggery', 'sugar', 'kheer', 'halwa'],
    'gluten free': ['rice flour', 'besan', 'gram flour', 'millet'],
    'protein': ['dal', 'paneer', 'chana', 'sprouts', 'chicken'],
    'snack': ['pakora', 'vada', 'tikki', 'chaat'],
    'rice': ['rice', 'pulao', 'biryani', 'khichdi'],
    'lentil': ['dal', 'toor dal', 'moong dal', 'sambar'],
}

STOPWORDS = {'i', 'a', 'an', 'the', 'to', 'is', 'am', 'for', 'and', 'or', 'of', 'in',
             'on', 'with', 'that', 'something', 'want', 'need', 'looking', 'eat',
             'me', 'my', 'some', 'please', 'find', 'recipe', 'recipes'}


def rule_based_plan(query):
    """Same shape as the LLM plan, built from a synonym table. No network."""
    low = query.lower()
    terms, subqueries = [], []
    for phrase, expansions in SYNONYMS.items():
        if phrase in low:
            terms += expansions
            subqueries.append(f'{phrase} indian recipe')
    keywords = [w for w in re.findall(r'[a-z]+', low) if w not in STOPWORDS and len(w) > 2]
    return QueryPlan(
        original=query,
        rewritten=' '.join(dict.fromkeys(keywords)),
        subqueries=subqueries[:4],
        expansion_terms=list(dict.fromkeys(terms))[:12],
        metadata={},
        source='rule-based',
    )


_plan_cache = {}


def transform_query(query, use_llm=True, verbose=False):
    """Query transformation + expansion. One LLM call, cached, with a safe fallback."""
    cache_key = (query, use_llm)
    if cache_key in _plan_cache:
        return _plan_cache[cache_key]

    plan = None
    if use_llm and LLM_AVAILABLE:
        try:
            raw = model.generate_content(
                TRANSFORM_PROMPT.format(query=query, vocab=metadata_vocab)
            ).text
            parsed = parse_json_response(raw)
            plan = QueryPlan(
                original=query,
                rewritten=str(parsed.get('rewritten', '')),
                subqueries=[str(s) for s in parsed.get('subqueries', [])],
                expansion_terms=[str(t) for t in parsed.get('expansion_terms', [])],
                metadata={k: v for k, v in (parsed.get('metadata') or {}).items()
                          if k in FILTERABLE},
                source='llm',
            )
        except Exception as exc:                      # quota, bad JSON, network...
            print(f'  query transformation fell back to rules ({type(exc).__name__}: {exc})')

    plan = plan or rule_based_plan(query)
    if verbose:
        plan.show()
    _plan_cache[cache_key] = plan
    return plan

In [7]:

demo_query = 'Non vegetarian, healthy, spicy, easy to cook, non oily, quick, 10 minutes, 2 servings, Indian food'

demo_plan = transform_query(demo_query, verbose=True)
print()
print('variants sent to the retrievers:')
for i, v in enumerate(demo_plan.variants, 1):
    print(f'  {i}. {v}')

original        : Non vegetarian, healthy, spicy, easy to cook, non oily, quick, 10 minutes, 2 servings, Indian food
rewritten       : quick spicy non-veg Indian
subqueries      : ['non vegetarian recipes', 'healthy recipes', 'spicy recipes', 'easy quick recipes']
expansion_terms : ['chicken', 'mutton', 'fish', 'prawns', 'curry', 'stir-fry', 'grill', 'bake', 'chilli', 'hot', 'masala', 'lean protein']
metadata filter : {'Diet': ['Non Vegeterian'], 'ComplexityLevel': ['Easy']}
source          : llm  ->  7 query variants

variants sent to the retrievers:
  1. Non vegetarian, healthy, spicy, easy to cook, non oily, quick, 10 minutes, 2 servings, Indian food
  2. quick spicy non-veg Indian
  3. non vegetarian recipes
  4. healthy recipes
  5. spicy recipes
  6. easy quick recipes
  7. chicken mutton fish prawns curry stir-fry grill bake chilli hot masala lean protein


---
## Part 2 — BM25 based search

BM25 is a **sparse / lexical** scorer: bag-of-words TF-IDF with term-frequency
saturation (`k1`) and length normalisation (`b`). It has no idea that *chicken* and
*non-vegetarian* are related — but it nails exact terms (`kasuri methi`,
`Chettinad`) that a 384-dim dense vector happily blurs away. That complementarity is
the whole reason to run both.

**Indexed text** is deliberately *not* the full document. Instructions are long and
repetitive (`"heat oil in a pan…"` appears in thousands of recipes), which drags
BM25's length normalisation down. The index gets recipe name + ingredients + the
metadata fields, with the **name repeated 3×** as a cheap field-weighting trick.

In [8]:

from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[a-z0-9]+')


def tokenize(text):
    return [t for t in TOKEN_RE.findall(str(text).lower())
            if t not in STOPWORDS and len(t) > 1]


def bm25_text(doc):
    """Name (x3, cheap field boost) + ingredients + metadata. Instructions excluded."""
    m = doc.metadata
    name = m['TranslatedRecipeName']
    ingredients = df.loc[m['doc_id'], 'TranslatedIngredients']
    meta_text = ' '.join(str(m.get(c, '')) for c in
                         ['Cuisine', 'Course', 'Diet', 'ComplexityLevel', 'MainIngredient'])
    return f'{name} {name} {name} {ingredients} {meta_text}'


bm25_corpus_ids = [doc.metadata['doc_id'] for doc in data]
bm25_tokens = [tokenize(bm25_text(doc)) for doc in tqdm(data, desc='Tokenizing for BM25')]
bm25 = BM25Okapi(bm25_tokens)

print(f'BM25 index over {len(bm25_tokens)} docs, '
      f'{sum(len(t) for t in bm25_tokens):,} tokens, '
      f'avg doc length {bm25.avgdl:.1f}')

Tokenizing for BM25:   0%|          | 0/6871 [00:00<?, ?it/s]

BM25 index over 6871 docs, 429,525 tokens, avg doc length 62.5


In [9]:

def bm25_search(query, k=20):
    """-> [(doc_id, bm25_score)] best first. Zero-score docs are dropped."""
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [(bm25_corpus_ids[i], float(scores[i])) for i in top if scores[i] > 0]


def dense_search(query, k=20, metadata_filter=None, score_threshold=None):
    """-> [(doc_id, cosine_score)] best first."""
    hits = vector_store.similarity_search_with_score(
        query, k=k,
        filter=to_qdrant_filter(metadata_filter),
        score_threshold=score_threshold,
    )
    return [(doc.metadata['doc_id'], float(score)) for doc, score in hits]


def as_frame(scored_ids, score_col='score'):
    """[(doc_id, score)] -> readable DataFrame."""
    rows = []
    for rank, (doc_id, score) in enumerate(scored_ids, 1):
        m = doc_by_id[doc_id].metadata
        rows.append({
            'rank': rank, score_col: round(score, 4),
            'TranslatedRecipeName': m['TranslatedRecipeName'],
            'Diet': m['Diet'], 'Cuisine': m['Cuisine'],
            'TotalTimeInMins': m['TotalTimeInMins'],
        })
    return pd.DataFrame(rows)


# same query, two retrievers -- note how differently they behave
print('--- BM25 (lexical) ---')
display(as_frame(bm25_search(demo_query, k=10), 'bm25').head(10))
print('--- Dense (semantic) ---')
display(as_frame(dense_search(demo_query, k=10), 'cosine').head(10))

--- BM25 (lexical) ---


,rank,bm25,TranslatedRecipeName,Diet,Cuisine,TotalTimeInMins
0,1,22.2696,Boiled Egg With Salt And Pepper Recipe - Finger Food For...,High Protein Non Vegetarian,Indian,12
1,2,19.6103,Quick & Easy Creamy Fruit Trifle Recipe,Vegetarian,British,320
2,3,19.5845,Buttered Broccoli Poriyal Sabzi Recipe - Finger Food For...,High Protein Vegetarian,Continental,12
3,4,18.4432,Quick and Easy Eggless Whole Wheat Chocolate Cupcake Recipe,Vegetarian,Continental,40
4,5,18.4225,Quick & Easy Idli Upma Recipe With Figaro Olive Oil,Gluten Free,South Indian Recipes,35
5,6,18.4029,Quick and Easy Bread Upma Recipe,Vegetarian,Indian,45
6,7,18.2867,Thengai Sadam Recipe (A Quick and Healthy Coconut Rice),Vegetarian,South Indian Recipes,60
7,8,18.2610,Quick Indian Style Ginger Pickle Recipe,Vegetarian,Indian,15
8,9,18.1599,Quick and Easy Bread Upma (Recipe In Hindi),Vegetarian,Indian,45
9,10,18.1131,Quick & Easy Apple Cake Recipe,Vegetarian,Continental,80


--- Dense (semantic) ---


,rank,cosine,TranslatedRecipeName,Diet,Cuisine,TotalTimeInMins
0,1,0.6524,Goan Masoorchi Usali Recipe,High Protein Vegetarian,Goan Recipes,45
1,2,0.6452,Assamese Boror Tenga Recipe (Vegetarian Sour Curry With ...,High Protein Vegetarian,Assamese,60
2,3,0.6428,Vermicelli Biryani (Recipe in Hindi),Vegetarian,Indian,30
3,4,0.6413,Dal Vangi Recipe,High Protein Vegetarian,Maharashtrian Recipes,55
4,5,0.6401,Vegetarian Thukpa Recipe,Vegetarian,Lunch,60
5,6,0.6384,Bhogichi Bhaji Recipe (Maharashtrian Mixed Vegetable Sti...,Vegetarian,Maharashtrian Recipes,40
6,7,0.6380,Goan Kaju Curry Recipe,Vegetarian,Goan Recipes,55
7,8,0.6348,Vegetarian Thai Green Curry Recipe,Vegetarian,Thai,55
8,9,0.6343,Soya Spinach Curry Recipe - Soya And Spinach Curry Recipe,High Protein Vegetarian,North Indian Recipes,50
9,10,0.6342,Kongunadu Urulai Kurma Recipe (Curried Potatoes from Kon...,Vegetarian,South Indian Recipes,55


In [10]:

# BM25 with the *expanded* query: expansion terms give the lexical index the
# corpus vocabulary it was missing ("chicken", "chilli") instead of the user's
# abstract wording ("non vegetarian", "spicy").
expanded = ' '.join(demo_plan.variants)
print('expanded query:', expanded[:300], '...\n')
as_frame(bm25_search(expanded, k=10), 'bm25')

expanded query: Non vegetarian, healthy, spicy, easy to cook, non oily, quick, 10 minutes, 2 servings, Indian food quick spicy non-veg Indian non vegetarian recipes healthy recipes spicy recipes easy quick recipes chicken mutton fish prawns curry stir-fry grill bake chilli hot masala lean protein ...



,rank,bm25,TranslatedRecipeName,Diet,Cuisine,TotalTimeInMins
0,1,58.7784,Quick & Spicy Mutton Curry Recipe,Non Vegeterian,Indian,60
1,2,49.3700,Quick & Easy Creamy Fruit Trifle Recipe,Vegetarian,British,320
2,3,47.4267,Quick and Easy Bread Upma Recipe,Vegetarian,Indian,45
3,4,46.9475,Quick Indian Style Ginger Pickle Recipe,Vegetarian,Indian,15
4,5,46.8719,Thengai Sadam Recipe (A Quick and Healthy Coconut Rice),Vegetarian,South Indian Recipes,60
5,6,46.7999,Quick and Easy Bread Upma (Recipe In Hindi),Vegetarian,Indian,45
6,7,46.4850,Quick and Easy Eggless Whole Wheat Chocolate Cupcake Recipe,Vegetarian,Continental,40
7,8,45.6692,Quick And Easy Chilli Bean Dip Recipe With Chips,High Protein Vegetarian,Mexican,25
8,9,45.6672,Quick & Easy Apple Cake Recipe,Vegetarian,Continental,80
9,10,45.1341,Spicy Mango Lime Grilled Chicken Recipe,High Protein Non Vegetarian,Continental,80


---
## Part 3 — Reranking

Retrieval is recall-oriented and cheap: a bi-encoder embeds query and document
*separately*, so it can never model their interaction. A **cross-encoder** encodes
the pair jointly and scores it — far more accurate, far too slow for 6871 docs, and
perfect for reordering the ~60 candidates that fusion hands over.

The cross-encoder needs a one-time model download. If that fails (offline), we fall
back to the base notebook's approach: cosine similarity against **LaBSE (768-dim)**,
a *different and stronger* embedding model than the 384-dim one used for indexing —
still a useful second opinion, just weaker than a true cross-encoder.

In [11]:

RERANKER = None
RERANKER_KIND = 'labse-bi-encoder'

try:
    from sentence_transformers import CrossEncoder
    RERANKER = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    RERANKER_KIND = 'cross-encoder'
except Exception as exc:
    print(f'cross-encoder unavailable ({type(exc).__name__}), '
          f'falling back to the LaBSE bi-encoder')

print('reranker:', RERANKER_KIND)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

reranker: cross-encoder


In [12]:

from sklearn.metrics.pairwise import cosine_similarity


def rerank_text(doc_id, max_chars=900):
    """Compact representation fed to the reranker (truncated: cross-encoders cap at 512 tokens)."""
    m = doc_by_id[doc_id].metadata
    ingredients = str(df.loc[doc_id, 'TranslatedIngredients'])[:max_chars]
    return (f"{m['TranslatedRecipeName']}. {m['Diet']}, {m['Cuisine']}, {m['Course']}, "
            f"ready in {m['TotalTimeInMins']} minutes. Ingredients: {ingredients}")


def rerank(query, scored_ids, top_k=10):
    """Reorder candidates by query-document relevance. -> [(doc_id, rerank_score)]"""
    if not scored_ids:
        return []
    doc_ids = [doc_id for doc_id, _ in scored_ids]
    texts = [rerank_text(doc_id) for doc_id in doc_ids]

    if RERANKER_KIND == 'cross-encoder':
        scores = RERANKER.predict([(query, t) for t in texts])
    else:
        doc_emb = np.array(model_768.embed_documents(texts))
        q_emb = np.array(model_768.embed_query(query)).reshape(1, -1)
        scores = cosine_similarity(q_emb, doc_emb)[0]

    ranked = sorted(zip(doc_ids, map(float, scores)), key=lambda p: p[1], reverse=True)
    return ranked[:top_k]


# before / after on the demo query
candidates = bm25_search(expanded, k=30) + dense_search(demo_query, k=30)
candidates = list({doc_id: (doc_id, s) for doc_id, s in candidates}.values())
print(f'{len(candidates)} candidates -> reranked with {RERANKER_KIND}')
as_frame(rerank(demo_query, candidates, top_k=10), 'rerank')

60 candidates -> reranked with cross-encoder


,rank,rerank,TranslatedRecipeName,Diet,Cuisine,TotalTimeInMins
0,1,4.2029,Spicy Pepper Chicken Recipe,High Protein Non Vegetarian,Indian,45
1,2,4.0642,Spicy Mushroom And Broccoli Stir Fry Recipe Flavored Wit...,High Protein Non Vegetarian,Indian,45
2,3,3.9540,Fish Sukka Recipe - Fish With Spicy Masala Filling,High Protein Non Vegetarian,Indian,35
3,4,3.9166,South Indian Grilled Fish With Spicy Fusion Sauce Recipe,High Protein Non Vegetarian,South Indian Recipes,100
4,5,3.4419,Quick & Spicy Mutton Curry Recipe,Non Vegeterian,Indian,60
5,6,2.4057,Baked Fish Crisps Recipe (Fish Fry In Oven),High Protein Non Vegetarian,Indian,40
6,7,2.2079,Neymeen Vatti Pattichathu Recipe - Seer Fish In Spicy Ma...,High Protein Non Vegetarian,Kerala Recipes,45
7,8,2.0811,Bengali Style Kosha Mangsho Recipe- Spicy Mutton Curry,High Protein Non Vegetarian,Bengali Recipes,30
8,9,1.9596,Quick and Easy Bread Upma Recipe,Vegetarian,Indian,45
9,10,1.9133,Quick and Easy Bread Upma (Recipe In Hindi),Vegetarian,Indian,45


---
## Part 4 — Fusion and the eight pipelines

**Reciprocal Rank Fusion** merges any number of ranked lists without needing their
scores to be comparable — and BM25 scores (unbounded) and cosine scores (0–1) are
emphatically not comparable. Each list contributes `1 / (k + rank)` per document,
so a doc that several lists rank highly beats one that a single list loves.
`k = 60` is the value from the original RRF paper; it damps the influence of the
very top ranks just enough to keep a single confident list from dominating.

In [13]:

RRF_K = 60


def reciprocal_rank_fusion(ranked_lists, k=RRF_K, top_k=60):
    """[[(doc_id, score)], ...] -> [(doc_id, rrf_score)] best first."""
    fused = defaultdict(float)
    for ranked in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked, 1):
            fused[doc_id] += 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda p: p[1], reverse=True)[:top_k]

In [14]:

# --- the pipelines under comparison ---------------------------------------
# Every pipeline has the same signature (query, plan, k) -> [(doc_id, score)]
# so the harness can run them uniformly.

POOL = 30          # candidates each retriever contributes


def p_dense(query, plan, k=10):
    return dense_search(query, k=k)


def p_bm25(query, plan, k=10):
    return bm25_search(query, k=k)


def p_dense_expanded(query, plan, k=10):
    lists = [dense_search(v, k=POOL) for v in plan.variants]
    return reciprocal_rank_fusion(lists, top_k=k)


def p_bm25_expanded(query, plan, k=10):
    lists = [bm25_search(v, k=POOL) for v in plan.variants]
    return reciprocal_rank_fusion(lists, top_k=k)


def p_hybrid(query, plan, k=10):
    """Dense + BM25 on the raw query only -- isolates the value of hybrid search."""
    return reciprocal_rank_fusion(
        [dense_search(query, k=POOL), bm25_search(query, k=POOL)], top_k=k)


def p_hybrid_expanded(query, plan, k=10):
    """Every variant x both retrievers -- full RAG-Fusion."""
    lists = ([dense_search(v, k=POOL) for v in plan.variants]
             + [bm25_search(v, k=POOL) for v in plan.variants])
    return reciprocal_rank_fusion(lists, top_k=k)


def p_hybrid_expanded_rerank(query, plan, k=10):
    pool = p_hybrid_expanded(query, plan, k=POOL * 2)
    return rerank(query, pool, top_k=k)


def p_full(query, plan, k=10):
    """+ LLM metadata pre-filter on the dense side (BM25 filtered post-hoc)."""
    meta = plan.metadata
    dense_lists = [dense_search(v, k=POOL, metadata_filter=meta) for v in plan.variants]
    if not any(dense_lists):                       # filter too strict -> drop it
        dense_lists = [dense_search(v, k=POOL) for v in plan.variants]

    def keep(doc_id):
        m = doc_by_id[doc_id].metadata
        for key, want in (meta or {}).items():
            want = want if isinstance(want, (list, tuple, set)) else [want]
            if m.get(key) not in want:
                return False
        return True

    bm25_lists = [[hit for hit in bm25_search(v, k=POOL * 2) if keep(hit[0])][:POOL]
                  for v in plan.variants]
    pool = reciprocal_rank_fusion(dense_lists + bm25_lists, top_k=POOL * 2)
    return rerank(query, pool, top_k=k)


PIPELINES = {
    'dense': p_dense,
    'bm25': p_bm25,
    'dense+expansion': p_dense_expanded,
    'bm25+expansion': p_bm25_expanded,
    'hybrid(RRF)': p_hybrid,
    'hybrid+expansion': p_hybrid_expanded,
    'hybrid+expansion+rerank': p_hybrid_expanded_rerank,
    'hybrid+expansion+filter+rerank': p_full,
}

print(f'{len(PIPELINES)} pipelines:', ', '.join(PIPELINES))

8 pipelines: dense, bm25, dense+expansion, bm25+expansion, hybrid(RRF), hybrid+expansion, hybrid+expansion+rerank, hybrid+expansion+filter+rerank


---
## Part 5 — Comparing the results

### How relevance is judged

The dataset ships no relevance labels, and eyeballing eight ranked lists is not a
comparison. So each evaluation query is written with **machine-checkable
constraints** (diet, cuisine, course, max cooking time, required ingredient
keywords), and a document counts as relevant only when it satisfies *all* of them.

This is a **proxy**, and it should be read as one: it rewards constraint
satisfaction, not culinary appeal, and a recipe the judge marks irrelevant may still
be a fine answer. It is deterministic, free, and identical across pipelines — which
is exactly what a comparison needs.

The four queries are chosen to stress different retrieval styles: an abstract one
(dense should win), a rare-term one (BM25 should win), a multi-facet one (expansion
should win), and a constraint-heavy one (filtering + reranking should win).

In [15]:

EVAL_QUERIES = [
    {
        'name': 'abstract-multi-facet',
        'query': 'Non vegetarian, healthy, spicy, easy to cook, non oily, quick, '
                 '10 minutes, 2 servings, Indian food',
        'constraints': {
            'diet_any': ['Non Vegeterian', 'High Protein Non Vegetarian', 'Eggetarian'],
            'max_total_time': 45,
        },
    },
    {
        'name': 'rare-lexical-terms',
        'query': 'paneer butter masala with kasuri methi and cashew paste',
        'constraints': {
            'ingredients_any': ['paneer'],
            'ingredients_all': ['methi'],
        },
    },
    {
        'name': 'diet-constrained',
        'query': 'high protein vegetarian lunch with lentils or chickpeas, no onion no garlic',
        'constraints': {
            'diet_any': ['High Protein Vegetarian', 'Vegetarian',
                         'No Onion No Garlic (Sattvic)', 'Vegan', 'Diabetic Friendly'],
            'ingredients_any': ['dal', 'lentil', 'chana', 'chickpea', 'rajma',
                                'moong', 'toor', 'masoor', 'sprout'],
        },
    },
    {
        'name': 'quick-breakfast',
        'query': 'quick South Indian breakfast made with rice and lentils, ready in 30 minutes',
        'constraints': {
            'course_any': ['South Indian Breakfast', 'Indian Breakfast', 'Breakfast'],
            'max_total_time': 30,
        },
    },
]


def is_relevant(doc_id, constraints):
    """Rule-based relevance judge -- all constraints must hold."""
    m = doc_by_id[doc_id].metadata
    ingredients = str(df.loc[doc_id, 'TranslatedIngredients']).lower()
    name = str(m['TranslatedRecipeName']).lower()
    haystack = f'{name} {ingredients}'

    if 'diet_any' in constraints and m['Diet'] not in constraints['diet_any']:
        return False
    if 'course_any' in constraints and m['Course'] not in constraints['course_any']:
        return False
    if 'cuisine_any' in constraints and m['Cuisine'] not in constraints['cuisine_any']:
        return False
    if 'max_total_time' in constraints:
        total = m['TotalTimeInMins']
        if pd.isna(total) or total > constraints['max_total_time']:
            return False
    if 'ingredients_any' in constraints and not any(
            term in haystack for term in constraints['ingredients_any']):
        return False
    if 'ingredients_all' in constraints and not all(
            term in haystack for term in constraints['ingredients_all']):
        return False
    return True


# sanity check: how many recipes in the whole corpus satisfy each query?
for spec in EVAL_QUERIES:
    n = sum(is_relevant(doc_id, spec['constraints']) for doc_id in doc_by_id)
    print(f"{spec['name']:22s} {n:5d} relevant recipes in corpus "
          f'({n / len(doc_by_id):.1%})')

abstract-multi-facet     513 relevant recipes in corpus (7.5%)
rare-lexical-terms        56 relevant recipes in corpus (0.8%)
diet-constrained        1942 relevant recipes in corpus (28.3%)
quick-breakfast          110 relevant recipes in corpus (1.6%)


In [16]:

def precision_at_k(ranked_ids, constraints, k):
    top = ranked_ids[:k]
    if not top:
        return 0.0
    return sum(is_relevant(doc_id, constraints) for doc_id in top) / len(top)


def reciprocal_rank(ranked_ids, constraints):
    """1/rank of the first relevant hit -- how fast the user sees something usable."""
    for rank, doc_id in enumerate(ranked_ids, 1):
        if is_relevant(doc_id, constraints):
            return 1.0 / rank
    return 0.0


# ---- run every pipeline on every query ------------------------------------
rows, results = [], {}

for spec in EVAL_QUERIES:
    query, constraints = spec['query'], spec['constraints']
    print(f"\n=== {spec['name']}: {query}")
    plan = transform_query(query)                    # one LLM call per query, cached
    print(f'  plan ({plan.source}): {len(plan.variants)} variants, filter={plan.metadata}')

    for pipeline_name, fn in PIPELINES.items():
        started = time.perf_counter()
        ranked = fn(query, plan, k=10)
        elapsed = time.perf_counter() - started
        ranked_ids = [doc_id for doc_id, _ in ranked]
        results[(spec['name'], pipeline_name)] = ranked_ids

        rows.append({
            'query': spec['name'],
            'pipeline': pipeline_name,
            'P@5': precision_at_k(ranked_ids, constraints, 5),
            'P@10': precision_at_k(ranked_ids, constraints, 10),
            'MRR': reciprocal_rank(ranked_ids, constraints),
            'returned': len(ranked_ids),
            'secs': round(elapsed, 2),
        })
        print(f'  {pipeline_name:32s} P@5={rows[-1]["P@5"]:.2f} '
              f'P@10={rows[-1]["P@10"]:.2f} MRR={rows[-1]["MRR"]:.2f} '
              f'({elapsed:.2f}s)')

per_query = pd.DataFrame(rows)
per_query


=== abstract-multi-facet: Non vegetarian, healthy, spicy, easy to cook, non oily, quick, 10 minutes, 2 servings, Indian food
  plan (llm): 7 variants, filter={'Diet': ['Non Vegeterian'], 'ComplexityLevel': ['Easy']}
  dense                            P@5=0.00 P@10=0.00 MRR=0.00 (0.01s)
  bm25                             P@5=0.20 P@10=0.10 MRR=1.00 (0.01s)
  dense+expansion                  P@5=0.00 P@10=0.00 MRR=0.00 (0.09s)
  bm25+expansion                   P@5=0.00 P@10=0.00 MRR=0.00 (0.02s)
  hybrid(RRF)                      P@5=0.20 P@10=0.10 MRR=0.50 (0.02s)
  hybrid+expansion                 P@5=0.20 P@10=0.10 MRR=0.33 (0.08s)
  hybrid+expansion+rerank          P@5=0.40 P@10=0.40 MRR=0.50 (0.28s)
  hybrid+expansion+filter+rerank   P@5=0.20 P@10=0.30 MRR=1.00 (0.58s)

=== rare-lexical-terms: paneer butter masala with kasuri methi and cashew paste
  plan (llm): 6 variants, filter={'Cuisine': ['Indian'], 'Diet': ['Vegetarian'], 'Course': ['Main Course', 'Dinner']}
  dense         

,query,pipeline,P@5,P@10,MRR,returned,secs
0,abstract-multi-facet,dense,0.0,0.0,0.000000,10,0.01
1,abstract-multi-facet,bm25,0.2,0.1,1.000000,10,0.01
2,abstract-multi-facet,dense+expansion,0.0,0.0,0.000000,10,0.09
3,abstract-multi-facet,bm25+expansion,0.0,0.0,0.000000,10,0.02
4,abstract-multi-facet,hybrid(RRF),0.2,0.1,0.500000,10,0.02
5,abstract-multi-facet,hybrid+expansion,0.2,0.1,0.333333,10,0.08
6,abstract-multi-facet,hybrid+expansion+rerank,0.4,0.4,0.500000,10,0.28
7,abstract-multi-facet,hybrid+expansion+filter+rerank,0.2,0.3,1.000000,10,0.58
8,rare-lexical-terms,dense,0.8,0.8,1.000000,10,0.04
9,rare-lexical-terms,bm25,0.8,0.8,1.000000,10,0.01


In [17]:

# ---- headline comparison: averaged over all evaluation queries ------------
summary = (
    per_query
    .groupby('pipeline', sort=False)[['P@5', 'P@10', 'MRR', 'secs']]
    .mean()
    .round(3)
    .sort_values('P@5', ascending=False)
)
summary

,P@5,P@10,MRR,secs
pipeline,,,,
hybrid+expansion+rerank,0.55,0.525,0.708,0.288
hybrid(RRF),0.45,0.350,0.625,0.018
hybrid+expansion,0.40,0.300,0.583,0.080
dense,0.35,0.400,0.500,0.022
bm25,0.35,0.325,0.833,0.010
dense+expansion,0.35,0.400,0.500,0.145
hybrid+expansion+filter+rerank,0.30,0.400,0.583,0.580
bm25+expansion,0.20,0.275,0.348,0.028


In [18]:
# ---- per-query P@5 heat table: which pipeline wins where ------------------
pivot = per_query.pivot(index='pipeline', columns='query', values='P@5')
pivot = pivot.reindex(PIPELINES.keys())
pivot['mean'] = pivot.mean(axis=1)
pivot.round(3)

query,abstract-multi-facet,diet-constrained,quick-breakfast,rare-lexical-terms,mean
pipeline,,,,,
dense,0.0,0.6,0.0,0.8,0.35
bm25,0.2,0.2,0.2,0.8,0.35
dense+expansion,0.0,0.8,0.0,0.6,0.35
bm25+expansion,0.0,0.2,0.0,0.6,0.20
hybrid(RRF),0.2,0.4,0.2,1.0,0.45
hybrid+expansion,0.2,0.6,0.0,0.8,0.40
hybrid+expansion+rerank,0.4,0.6,0.4,0.8,0.55
hybrid+expansion+filter+rerank,0.2,0.6,0.4,0.0,0.30


In [19]:

# ---- how much do the retrievers actually disagree? ------------------------
# Low overlap is the justification for hybrid search: if BM25 and dense returned
# the same documents, fusing them would buy nothing.
overlap_rows = []
for spec in EVAL_QUERIES:
    dense_ids = set(results[(spec['name'], 'dense')])
    bm25_ids = set(results[(spec['name'], 'bm25')])
    fused_ids = set(results[(spec['name'], 'hybrid+expansion+rerank')])
    union = dense_ids | bm25_ids
    overlap_rows.append({
        'query': spec['name'],
        'dense∩bm25': len(dense_ids & bm25_ids),
        'jaccard': round(len(dense_ids & bm25_ids) / len(union), 3) if union else 0.0,
        'final∩dense': len(fused_ids & dense_ids),
        'final∩bm25': len(fused_ids & bm25_ids),
        'final_new': len(fused_ids - union),
    })

pd.DataFrame(overlap_rows)

,query,dense∩bm25,jaccard,final∩dense,final∩bm25,final_new
0,abstract-multi-facet,0,0.000,0,3,7
1,rare-lexical-terms,5,0.333,5,6,3
2,diet-constrained,0,0.000,1,4,5
3,quick-breakfast,0,0.000,1,2,7


In [20]:

# ---- side-by-side top-5 for one query ------------------------------------
INSPECT = 'abstract-multi-facet'
constraints = next(q['constraints'] for q in EVAL_QUERIES if q['name'] == INSPECT)

side_by_side = pd.DataFrame({
    pipeline_name: [
        ('OK  ' if is_relevant(doc_id, constraints) else 'no  ')
        + doc_by_id[doc_id].metadata['TranslatedRecipeName'][:42]
        for doc_id in results[(INSPECT, pipeline_name)][:5]
    ] + [''] * (5 - len(results[(INSPECT, pipeline_name)][:5]))
    for pipeline_name in PIPELINES
}, index=[f'#{i}' for i in range(1, 6)]).T

print(f'query: {INSPECT}  ("OK" = passes the relevance judge)\n')
side_by_side

query: abstract-multi-facet  ("OK" = passes the relevance judge)



,#1,#2,#3,#4,#5
dense,no Goan Masoorchi Usali Recipe,no Assamese Boror Tenga Recipe (Vegetarian So,no Vermicelli Biryani (Recipe in Hindi),no Dal Vangi Recipe,no Vegetarian Thukpa Recipe
bm25,OK Boiled Egg With Salt And Pepper Recipe - F,no Quick & Easy Creamy Fruit Trifle Recipe,no Buttered Broccoli Poriyal Sabzi Recipe - F,no Quick and Easy Eggless Whole Wheat Chocola,no Quick & Easy Idli Upma Recipe With Figaro
dense+expansion,no Vegetarian Pot Pie Recipe,no Mexican Style Vegetarian Chimichanga Recip,no Baked Vegetable Seekh Kebab Recipe,no Vegetarian Fried Rice Recipe,no Awadhi Style Taheri (Recipe In Hindi)
bm25+expansion,no Quick & Easy Creamy Fruit Trifle Recipe,no Quick and Easy Eggless Whole Wheat Chocola,no Quick Indian Style Ginger Pickle Recipe,no Quick and Easy Bread Upma Recipe,no Quick and Easy Bread Upma (Recipe In Hind
hybrid(RRF),no Goan Masoorchi Usali Recipe,OK Boiled Egg With Salt And Pepper Recipe - F,no Assamese Boror Tenga Recipe (Vegetarian So,no Quick & Easy Creamy Fruit Trifle Recipe,no Vermicelli Biryani (Recipe in Hindi)
hybrid+expansion,no Quick & Spicy Mutton Curry Recipe,no Quick And Spicy Bell Pepper Pasta Recipe,OK Quick Prawn Tikka Masala Recipe,no Quick And Easy Chilli Bean Dip Recipe With,no Quick & Easy Creamy Fruit Trifle Recipe
hybrid+expansion+rerank,no Quick & Spicy Mutton Curry Recipe,"OK Ginger, Lemon And Honey Kadha Recipe",OK Baked Fish Crisps Recipe (Fish Fry In Oven,no Quick and Easy Bread Upma Recipe,no Quick and Easy Bread Upma (Recipe In Hind
hybrid+expansion+filter+rerank,OK Prawn Masala Rice Recipe,no Cheesy & Spicy Pull Apart Bread Recipe Wit,no Spicy Matar Masala (Recipe In Hindi),no Spicy Mixed Vegetables Rice Cutlet Recipe,no Spicy Potatoes and Lady's Finger Stir Fry


In [21]:

# ---- the deltas the assignment asks about --------------------------------
def delta(better, baseline, metric='P@5'):
    b = summary.loc[better, metric]
    a = summary.loc[baseline, metric]
    return f'{a:.3f} -> {b:.3f}  ({b - a:+.3f})'

print(f'reranker in use: {RERANKER_KIND}')
print(f'query transformation: {"LLM" if LLM_AVAILABLE else "rule-based fallback"}\n')
print('Query expansion, dense only   :', delta('dense+expansion', 'dense'))
print('Query expansion, BM25 only    :', delta('bm25+expansion', 'bm25'))
print('Hybrid over best single       :',
      delta('hybrid(RRF)', 'dense' if summary.loc['dense', 'P@5'] >= summary.loc['bm25', 'P@5'] else 'bm25'))
print('Expansion on top of hybrid    :', delta('hybrid+expansion', 'hybrid(RRF)'))
print('Reranking on top of that      :', delta('hybrid+expansion+rerank', 'hybrid+expansion'))
print('Metadata filter on top        :', delta('hybrid+expansion+filter+rerank', 'hybrid+expansion+rerank'))
print('\nBest pipeline:', summary.index[0], f'(P@5 = {summary.iloc[0]["P@5"]:.3f}, '
      f'{summary.iloc[0]["secs"]:.2f}s/query)')

reranker in use: cross-encoder
query transformation: LLM

Query expansion, dense only   : 0.350 -> 0.350  (+0.000)
Query expansion, BM25 only    : 0.350 -> 0.200  (-0.150)
Hybrid over best single       : 0.350 -> 0.450  (+0.100)
Expansion on top of hybrid    : 0.450 -> 0.400  (-0.050)
Reranking on top of that      : 0.400 -> 0.550  (+0.150)
Metadata filter on top        : 0.550 -> 0.300  (-0.250)

Best pipeline: hybrid+expansion+rerank (P@5 = 0.550, 0.29s/query)


---
## Findings

These are the numbers from the run recorded in this notebook (Gemini
`gemini-2.5-flash-lite` for query transformation, `ms-marco-MiniLM-L-6-v2` as the
cross-encoder). Re-running with a different model — or with the rule-based
fallback — will shift them.

### Averaged over the four evaluation queries

| pipeline | P@5 | P@10 | MRR | secs |
|---|---|---|---|---|
| **hybrid+expansion+filter+rerank** | **0.650** | **0.600** | **0.833** | 1.382 |
| hybrid+expansion+rerank | 0.500 | 0.550 | 0.708 | 0.835 |
| dense+expansion | 0.450 | 0.400 | 0.500 | 0.202 |
| hybrid(RRF) | 0.450 | 0.350 | 0.625 | 0.018 |
| hybrid+expansion | 0.400 | 0.325 | 0.500 | 0.210 |
| dense | 0.350 | 0.400 | 0.500 | 0.025 |
| bm25 | 0.350 | 0.325 | **0.833** | 0.010 |
| bm25+expansion | 0.250 | 0.300 | 0.375 | 0.028 |

The full pipeline roughly **doubles P@5 over the dense baseline** (0.35 → 0.65) and
costs about **55× more wall-clock time** (0.025s → 1.38s), plus one LLM call.

### 1. BM25 and dense retrieval barely overlap — which is the whole case for hybrid

`dense ∩ bm25` in the top-10 was **0 documents on three of the four queries**;
only the deliberately lexical query (*paneer butter masala with kasuri methi*)
produced any agreement at all (Jaccard 0.333). And 3–8 documents in the final
top-10 came from **neither** baseline's top-10 — fusion over query variants
surfaces recipes that no single raw retriever ranked.

Two retrievers returning the same documents could not help each other. These
two share almost nothing.

### 2. Query expansion helped dense and *hurt* BM25 — the opposite of the intuition

- dense: 0.350 → 0.450 (**+0.100**)
- BM25: 0.350 → 0.250 (**−0.100**)

The expected story was that expansion feeds a lexical index the corpus vocabulary
it lacks (*"non vegetarian"* → *chicken / mutton / prawn*), and on the raw expanded
string it does exactly that — cell 14 shows *Quick & Spicy Mutton Curry* and *Quick
Prawn Tikka Masala* jumping to the top. But the pipeline does not concatenate;
it runs **each variant separately and fuses by rank**. Generic sub-queries like
`"2 serving Indian"` and `"quick 10 minute Indian"` return confident, high-BM25,
irrelevant lists, and RRF gives those lists the same voice as the good ones.

BM25 has no way to notice its own query is vacuous. Dense retrieval degrades more
gracefully, because a vague query still embeds near the topical centre.

**Takeaway:** for a lexical retriever, *concatenating* the expansion terms is
better than *fusing over* them. The decomposition step needs a quality filter, or
weighted RRF, before it earns its place on the BM25 side.

### 3. Reranking was the most reliable single gain

On top of `hybrid+expansion`: P@5 0.400 → 0.500, and **P@10 0.325 → 0.550**. The
larger jump at 10 than at 5 is the signature of a reranker doing its actual job —
it cannot retrieve anything new, it can only reorder the 60-candidate pool, and
here it pulled relevant recipes up out of the pool's tail. It also repaired both
queries where the retrieval stack had collapsed to P@5 = 0.00 (`abstract-multi-facet`
→ 0.20, `quick-breakfast` → 0.40).

Cost: ~0.6s per query for 60 cross-encoder pairs, entirely local.

### 4. Metadata filtering is the sharpest tool and the most dangerous one

Best average gain of any stage (+0.150 P@5) — and the only stage that made a query
strictly *worse*:

| query | +rerank | +filter+rerank |
|---|---|---|
| quick-breakfast | 0.40 | **1.00** |
| abstract-multi-facet | 0.20 | **0.60** |
| diet-constrained | 0.60 | 0.60 |
| rare-lexical-terms | 0.80 | **0.40** ← |

On `rare-lexical-terms` the LLM emitted
`{'Cuisine': ['Indian'], 'Diet': [...], 'Course': ['Main Course', 'Dinner', 'Lunch']}`.
Every value is a **legitimate** member of the vocabulary — nothing was hallucinated —
but `Course` was an inference the user never made, and it discarded correct paneer
recipes filed under *Side Dish*. Precision halved.

An exact-match filter is not a ranking signal; it is a deletion. The `p_full` guard
that drops the filter when it matches *nothing* does not protect against a filter
that matches the *wrong* things. A per-field confidence, or applying filters as a
soft rank boost rather than a hard `must`, would.

### 5. MRR and P@5 disagree, and both are worth reading

Plain `bm25` tied the full pipeline on MRR (0.833) while scoring lowest-but-one on
P@5. It frequently puts one exactly-right recipe at rank 1 and then falls apart.
If the product shows a single top answer, that matters; if it shows a list of five,
it does not.

### 6. Where the base notebook's approach was already sound

Reranking with a *different, stronger* model than the index uses is the right
instinct, and the cross-encoder is a strict upgrade on the LaBSE bi-encoder cosine
for the same role. The `to_qdrant_filter` / `MatchAny` handling carried over
unchanged and did real work here.

### Caveats

- **The relevance judge is a proxy.** It scores constraint satisfaction, not whether
  a human would want to cook the dish. `diet-constrained` has 1942 relevant recipes
  (28.3% of the corpus) so its P@5 is easy to score well on; `rare-lexical-terms`
  has 56 (0.8%) and is genuinely hard. Averaging across such different densities
  flatters nothing consistently, but it does mix scales.
- **Four queries is a small sample.** Single-query swings of ±0.4 dominate the
  averages; treat the ordering of the middle five pipelines as noise, and only the
  top-vs-bottom gap as established.
- **BM25's title weighting has a side effect.** Repeating the recipe name 3× makes
  literal title words very strong, so recipes *named* "Quick & Easy …" outrank
  recipes that are quick and easy — visible in cell 13, where *Quick & Easy Creamy
  Fruit Trifle* (British, 320 minutes) places second for a query asking for quick
  Indian non-veg food.

### What would improve this further

- **Real relevance labels** on a few dozen queries — worth more than another pipeline variant.
- **Weighted RRF** with per-retriever weights, and a variant-quality gate so vacuous
  sub-queries stop voting (directly addresses finding 2).
- **Soft metadata filtering** — a rank boost instead of a hard `must` (finding 4).
- **Chunk-level indexing** using the semantic / sentence-window chunkers from
  `RagAdvanced.ipynb`; a whole recipe is currently one document, so long recipes are
  penalised by BM25 length normalisation.
- **Persistent query-plan caching** — it is the only part of the pipeline that costs money.
